# 02 - Métricas e comparação fake e real
CRISP-NET, fases 4 (Modelagem) e 5 (Avaliação). Teste de Mann-Whitney com ajuste de Holm (Metodologia p.4 e p.37).

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import networkx as nx
import pandas as pd
from src.carregamento import carregar_upfd
from src.cascata import montar_cascatas

vertices, arestas, arvores = carregar_upfd()
cascatas = montar_cascatas(vertices, arestas, arvores)
EXCLUIDAS = {"politifact14940": "aparece nas listas fake e real do FakeNewsNet"}
validas = {k: G for k, G in cascatas.items() if G.graph["noticia"] not in EXCLUIDAS}
print(len(cascatas), "cascatas carregadas;", len(validas), "após exclusões")

In [ ]:
from src.metricas import METRICAS, calcular_metricas
from src.estatistica import comparar_grupos
from src.visualizacao import boxplots_metricas, desenhar_cascata

tabela = calcular_metricas(validas)
tabela.head()

## Comparação estatística

In [ ]:
comparacao = comparar_grupos(tabela, list(METRICAS))
comparacao.round(4)

## Distribuições

In [ ]:
boxplots_metricas(tabela, list(METRICAS),
                  escala_log={"tamanho", "largura_maxima", "grau_max_usuario", "tempo_ate_metade_h"});

## Mesmo tamanho, formas diferentes
Uma cascata fake e uma real com cerca de 100 vértices.

In [ ]:
import matplotlib.pyplot as plt
fig, eixos = plt.subplots(1, 2, figsize=(14, 5))
for ax, rotulo in zip(eixos, ("fake", "real")):
    sub = tabela[tabela["rotulo"] == rotulo]
    arvore = int(sub.loc[(sub["tamanho"] - 100).abs().idxmin(), "arvore"])
    desenhar_cascata(validas[arvore], ax=ax)